In [0]:
%pip install pyyaml openpyxl requests

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path
from datetime import date

CODE_ROOT = Path(
    "/Workspace/Users/arfaashaikh101@gmail.com/retail_dev_dbx"
)

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from scripts.bronze_loader import (
    load_config,
    prepare_source,
    ingest_range,
    table,
)

from scripts.transformations import create_layer_tables

config = load_config(
    spark,
    CODE_ROOT / "config/source_contracts.yaml",
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

TRIGGER_TYPE = "MANUAL"

# False: process only the first monthly batch for the initial test.
# True: process all historical monthly batches.
LOAD_ALL_HISTORY = False

In [0]:
env = config["environment"]

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {config['namespace']}"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"{config['namespace']}.{env['volume']}"
)

create_layer_tables(
    spark,
    config,
    {
        "bronze": "bronze_transactions",
        "quarantine": "retail_quarantine",
        "monitoring": "retail_monitoring",
    },
)

In [0]:
source_path, source_manifest = prepare_source(config)

print("Source:", source_path)
print("Original rows:", source_manifest["row_count"])

assert source_manifest["row_count"] == 1067371

In [0]:
def next_month(day):
    if day.month == 12:
        return date(day.year + 1, 1, 1)

    return date(day.year, day.month + 1, 1)


start = date.fromisoformat(
    config["source"]["start_date"]
)

end = date.fromisoformat(
    config["source"]["historical_end_exclusive"]
)

batches = []
cursor = start

while cursor < end:
    batch_end = min(next_month(cursor), end)

    batches.append({
        "start": cursor.isoformat(),
        "end": batch_end.isoformat(),
    })

    cursor = batch_end

selected_batches = (
    batches if LOAD_ALL_HISTORY else batches[:1]
)

print("Batches selected:", len(selected_batches))

for batch in selected_batches:
    print(batch)

In [0]:
results = []

for batch in selected_batches:
    run_id, status = ingest_range(
        spark=spark,
        config=config,
        source_path=source_path,
        source_manifest=source_manifest,
        start=batch["start"],
        end=batch["end"],
        load_type="HISTORICAL",
        trigger_type=TRIGGER_TYPE,
        refresh=False,
    )

    result = {
        **batch,
        "run_id": run_id,
        "status": status,
    }

    results.append(result)
    print(result)

print("Bronze execution completed")